In [44]:
import sys
from pathlib import Path
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import stats
from ipywidgets import interact, Dropdown, IntSlider, SelectionRangeSlider
sys.path.insert(0, '../')
from ultility.metrics import rolling_std_vol

In [45]:
for csv_name, key in [('VN30_INDEX.csv', 'VN30 Index'), ('VN_INDEX.csv', 'VN Index')]:
    df_temp = pd.read_csv(f'../dataset/{csv_name}')
    df_temp['time'] = pd.to_datetime(df_temp['time'], format='mixed', dayfirst=True, errors='coerce')
    df_temp = df_temp.sort_values('time')
    ret_col = 'return_1_day'
    df_temp_filtered = df_temp[df_temp['time'].dt.year >= 2010]
    null_count = df_temp_filtered[ret_col].isna().sum()
    if null_count > 0:
        for idx, val in df_temp_filtered[df_temp_filtered[ret_col].isna()][ret_col].items():
            print(f"{key}: null at {df_temp.loc[idx, 'time'].strftime('%Y-%m-%d')}")
    datasets[key] = df_temp_filtered.set_index('time')[ret_col] * 100

for file in ['DAX_40.csv', 'EuroNext_100.csv', 'IBEX_35.csv', 'KOSPI_index.csv', 'SMI.csv', 'snp500.csv', 'Nikkei_225.csv']:
    df_temp = pd.read_csv(f'../dataset/{file}')
    if 'Date' in df_temp.columns:
        df_temp.rename(columns={'Date': 'time'}, inplace=True)
    df_temp['time'] = pd.to_datetime(df_temp['time'], format='mixed', dayfirst=False, errors='coerce')
    df_temp = df_temp.dropna(subset=['time']).sort_values('time')
    ret_col = 'return_1_day'
    df_temp_filtered = df_temp[df_temp['time'].dt.year >= 2010]
    null_count = df_temp_filtered[ret_col].isna().sum()
    if null_count > 0:
        for idx, val in df_temp_filtered[df_temp_filtered[ret_col].isna()][ret_col].items():
            print(f"{file}: null at {df_temp.loc[idx, 'time'].strftime('%Y-%m-%d')}")
    datasets[file.replace('.csv', '')] = df_temp_filtered.set_index('time')[ret_col] * 100


In [46]:
summary_split = pd.DataFrame({
    "Dataset": [
        "VN30 Index", "VN Index", "DAX_40", "EuroNext_100", "IBEX_35", "KOSPI_index", "SMI", "snp500", "Nikkei_225"
    ],
    "train_size": [1971, 1964, 1983, 2005, 1815, 1944, 1987, 1988, 1677],
    "val_size":   [1096, 1103, 1104, 1115, 1362, 1109, 1135, 1109, 1174],
    "test_size":  [925, 925, 972, 979, 923, 881, 901, 927, 1062],
    "split_i":    [1971, 1964, 1983, 2005, 1815, 1944, 1987, 1988, 1677],
    "split_j":    [3067, 3067, 3087, 3120, 3177, 3053, 3122, 3097, 2851]
})
split_df = summary_split.set_index("Dataset")[["train_size", "val_size", "test_size"]]
split_df.columns = ["Train", "Val", "Test"]
split_df

,Train,Val,Test
Dataset,,,
VN30 Index,1971,1096,925
VN Index,1964,1103,925
DAX_40,1983,1104,972
EuroNext_100,2005,1115,979
IBEX_35,1815,1362,923
KOSPI_index,1944,1109,881
SMI,1987,1135,901
snp500,1988,1109,927
Nikkei_225,1677,1174,1062


In [47]:
def calc_vol(x, typ, window):
    if typ == "abs(return)":
        return x.abs()
    if typ == "vol(rolling)":
        return x.rolling(window).std()
    return x.rolling(window).apply(lambda y: (y**2).mean()**0.5, raw=True)

def plot_chart(ds, chart_type, typ, period, window):
    try:
        series = datasets[ds]
        tr, va, te = map(int, split_df.loc[ds, ['Train', 'Val', 'Test']])
        fig = go.Figure()
        if chart_type == "Volatility":
            full_v = calc_vol(series, typ, window)
            if period == 'all':
                for s, e, c, n in [(0, tr, '#1f77b4', 'train'), (tr, tr+va, '#ff7f0e', 'val'), (tr+va, tr+va+te, '#2ca02c', 'test')]:
                    fig.add_trace(go.Scatter(x=series.index[s:e], y=full_v.iloc[s:e], mode="lines", name=n, line=dict(color=c), connectgaps=False))
            else:
                s, e = (0, tr) if period == 'train' else (tr, tr+va) if period == 'val' else (tr+va, tr+va+te)
                fig.add_trace(go.Scatter(x=series.index[s:e], y=full_v.iloc[s:e], mode="lines", name=period, line=dict(color="#1f77b4"), connectgaps=False))
            fig.update_layout(title=f"{ds} - {typ} ({period}, window={window})", xaxis_title="Time", yaxis_title="Volatility")
        else:
            if period == 'all':
                for s, e, n in [(0, tr, 'train'), (tr, tr+va, 'val'), (tr+va, tr+va+te, 'test')]:
                    fig.add_trace(go.Histogram(x=series.iloc[s:e], nbinsx=50, name=n))
            else:
                s, e = (0, tr) if period == 'train' else (tr, tr+va) if period == 'val' else (tr+va, tr+va+te)
                fig.add_trace(go.Histogram(x=series.iloc[s:e], nbinsx=50, name=ds))
            fig.update_layout(title=f"{ds} - Return Distribution ({period})", xaxis_title="Return (%)", yaxis_title="Frequency", xaxis=dict(range=[-16, 16]))
        fig.update_layout(height=500)
        fig.show()
    except Exception as e:
        print(f"Error: {e}")

ds_opts = [k for k in split_df.index if k in datasets]
interact(
    plot_chart,
    ds=Dropdown(options=ds_opts),
    chart_type=Dropdown(options=["Volatility", "Histogram"], value="Volatility"),
    typ=Dropdown(options=["abs(return)", "vol(rolling)", "realized_vol"], value="vol(rolling)"),
    period=Dropdown(options=["train", "val", "test", "all"], value="train"),
    window=Dropdown(options=[10, 20, 30, 50, 60, 120, 250], value=60)
)

interactive(children=(Dropdown(description='ds', options=('VN30 Index', 'VN Index', 'DAX_40', 'EuroNext_100', …

<function __main__.plot_chart(ds, chart_type, typ, period, window)>

In [48]:
preds_df = pd.read_csv('../output/predictions_2010_2025.csv')
preds_df['Date'] = pd.to_datetime(preds_df['Date'])
preds_df = preds_df[preds_df['Model'] != 'TransformerGARCH']
nu_dict = dict(zip(dist_df['Dataset'], dist_df['nu_train']))

metrics_list = []
for ds in split_df.index:
    tr, va, te = int(split_df.loc[ds, 'Train']), int(split_df.loc[ds, 'Val']), int(split_df.loc[ds, 'Test'])
    series = datasets[ds]
    test_start = tr + va
    test_data = series.iloc[test_start:test_start+te].values
    test_dates = series.index[test_start+60:test_start+te]
    realized_vol_vals = rolling_std_vol(test_data, 60)[60:]
    
    for model in preds_df['Model'].unique():
        model_preds = preds_df[(preds_df['Dataset'] == ds) & (preds_df['Model'] == model)]['predicted_vol'].values
        if len(model_preds) == 0 or len(realized_vol_vals) < len(model_preds):
            continue
        realized_vol_model = realized_vol_vals[:len(model_preds)]
        
        mse = np.mean((realized_vol_model - model_preds) ** 2)
        qlike = np.mean(np.log(model_preds**2 + 1e-8) + realized_vol_model**2 / (model_preds**2 + 1e-8))
        
        nu = nu_dict.get(ds, 5)
        var_95_q = np.percentile(test_data, 5)
        t_alpha = stats.t.ppf(0.05, df=nu)
        var_95_dist = t_alpha * model_preds * np.sqrt((nu - 2) / nu)
        
        returns_eval = test_data[-len(model_preds):]
        viol_q = (returns_eval < var_95_q).mean()
        viol_d = (returns_eval < -var_95_dist).mean()
        
        kupiec_lr, kupiec_p = np.nan, np.nan
        n_viol = (returns_eval < -var_95_dist).sum()
        if n_viol > 0 and n_viol < len(returns_eval):
            viol_rate = n_viol / len(returns_eval)
            lr_stat = -2 * ((len(returns_eval)-n_viol)*np.log(1-0.05) + n_viol*np.log(0.05) - (len(returns_eval)-n_viol)*np.log(1-viol_rate) - n_viol*np.log(viol_rate))
            kupiec_p = 1 - stats.chi2.cdf(lr_stat, 1)
            kupiec_lr = lr_stat
        
        metrics_list.append({'Dataset': ds, 'Model': model, 'MSE': mse, 'QLIKE': qlike, 'Violation_Rate_95%': viol_d, 'Kupiec_LR': kupiec_lr, 'Kupiec_p': kupiec_p})

metrics_df = pd.DataFrame(metrics_list)
print(metrics_df.to_string(index=False))

     Dataset       Model      MSE         QLIKE  Violation_Rate_95%   Kupiec_LR  Kupiec_p
  VN30 Index       GARCH 0.221542      1.574481            0.931792 4404.439398       0.0
  VN30 Index   GJR-GARCH 0.230934      1.577048            0.932948 4415.575654       0.0
  VN30 Index        LSTM 1.317050 184981.832553            0.534104 1614.281797       0.0
  VN30 Index Transformer 1.506656 118377.678556            0.472832 1300.698877       0.0
  VN30 Index   LSTMGARCH 0.590828      1.622878            0.967630 4770.535150       0.0
    VN Index       GARCH 0.214718      1.449967            0.944509 4529.078222       0.0
    VN Index   GJR-GARCH 0.214450      1.450125            0.944509 4529.078222       0.0
    VN Index        LSTM 1.019039  66474.105480            0.578035 1855.187498       0.0
    VN Index Transformer 1.313196   4302.464651            0.478613 1329.171394       0.0
    VN Index   LSTMGARCH 0.559144      1.504116            0.972254 4821.891471       0.0
      DAX_

In [52]:
metrics_df.to_csv("../output/metrics.csv", index=False)

In [ ]:
def plot_vol_detailed(model, dataset):
    """Compare predicted vol, realized vol (60-day), and realized vol (10-day)"""
    tr, va, te = int(split_df.loc[dataset, 'Train']), int(split_df.loc[dataset, 'Val']), int(split_df.loc[dataset, 'Test'])
    series = datasets[dataset]
    test_start = tr + va
    test_data = series.iloc[test_start:test_start+te].values
    
    # Realized volatility (60-day)
    realized_vol_60 = rolling_std_vol(test_data, 60)[60:]
    
    # Realized volatility (10-day)
    realized_vol_10 = rolling_std_vol(test_data, 10)[60:]
    
    # Model predictions
    model_preds = preds_df[(preds_df['Dataset'] == dataset) & (preds_df['Model'] == model)]['predicted_vol'].values
    if len(model_preds) == 0:
        print(f'No predictions for {model} on {dataset}')
        return
    
    model_dates = preds_df[(preds_df['Dataset'] == dataset) & (preds_df['Model'] == model)]['Date'].values[:len(model_preds)]
    
    # Align lengths
    min_len = min(len(realized_vol_60), len(realized_vol_10), len(model_preds))
    realized_vol_60 = realized_vol_60[:min_len]
    realized_vol_10 = realized_vol_10[:min_len]
    model_preds = model_preds[:min_len]
    model_dates = model_dates[:min_len]
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=model_dates, y=model_preds, name=f'{model} (Predicted)', mode='lines', line=dict(color='blue', width=2)))
    fig.add_trace(go.Scatter(x=model_dates, y=realized_vol_60, name='Realized Vol (60-day)', mode='lines', line=dict(color='red', width=2)))
    fig.add_trace(go.Scatter(x=model_dates, y=realized_vol_10, name='Realized Vol (10-day)', mode='lines', line=dict(color='orange', width=1), opacity=0.8))
    
    fig.update_layout(
        title=f'{model} vs Realized Volatility - {dataset}',
        xaxis_title='Date',
        yaxis_title='Volatility',
        hovermode='x unified',
        height=600,
        template='plotly_white'
    )
    fig.show()

model_opts = sorted(preds_df['Model'].unique())
dataset_opts = sorted(preds_df['Dataset'].unique())
interact(plot_vol_detailed, model=Dropdown(options=model_opts, value=model_opts[0]), dataset=Dropdown(options=dataset_opts, value=dataset_opts[0]))

interactive(children=(Dropdown(description='model', options=('GARCH', 'GJR-GARCH', 'LSTM', 'LSTMGARCH', 'Trans…

<function __main__.plot_vol_detailed(model, dataset)>

In [56]:
def plot_var_backtest(model, dataset):
    """VaR Backtest Pipeline:
    1. Get predicted vol from model
    2. Calculate VaR threshold = t_alpha * predicted_vol * sqrt((nu-2)/nu)
    3. Compare actual returns vs VaR threshold
    4. Mark violations
    """
    tr, va, te = int(split_df.loc[dataset, 'Train']), int(split_df.loc[dataset, 'Val']), int(split_df.loc[dataset, 'Test'])
    series = datasets[dataset]
    test_start = tr + va
    test_data = series.iloc[test_start:test_start+te].values
    
    # Get model predictions
    model_preds = preds_df[(preds_df['Dataset'] == dataset) & (preds_df['Model'] == model)]['predicted_vol'].values
    if len(model_preds) == 0:
        print(f'No predictions for {model} on {dataset}')
        return
    
    model_dates = preds_df[(preds_df['Dataset'] == dataset) & (preds_df['Model'] == model)]['Date'].values[:len(model_preds)]
    
    # Align data
    returns_eval = test_data[-len(model_preds):]
    
    # Calculate VaR threshold: using t-distribution
    nu = nu_dict.get(dataset, 5)
    t_alpha = stats.t.ppf(0.05, df=nu)  # 5% quantile
    var_95_dist = t_alpha * model_preds * np.sqrt((nu - 2) / nu)
    
    # Identify violations (returns < VaR threshold means loss > threshold)
    violations = returns_eval < var_95_dist
    violation_indices = np.where(violations)[0]
    
    fig = go.Figure()
    
    # Plot actual returns
    fig.add_trace(go.Scatter(x=model_dates, y=returns_eval, name='Actual Returns', 
                            mode='markers', marker=dict(color='black', size=4)))
    
    # Plot VaR threshold (upper bound, should not exceed)
    fig.add_trace(go.Scatter(x=model_dates, y=var_95_dist, name='5% VaR Threshold',
                            mode='lines', line=dict(color='red', width=2, dash='dash')))
    
    # Plot predicted volatility as reference
    fig.add_trace(go.Scatter(x=model_dates, y=model_preds, name='Predicted Vol',
                            mode='lines', line=dict(color='blue', width=1, dash='dot'), opacity=0.6))
    
    # Highlight violations
    if len(violation_indices) > 0:
        fig.add_trace(go.Scatter(x=model_dates[violation_indices], 
                                y=returns_eval[violation_indices],
                                name=f'Violations ({len(violation_indices)})',
                                mode='markers', 
                                marker=dict(color='red', size=8, symbol='X')))
    
    # Add statistics
    violation_rate = violations.mean() * 100
    fig.add_annotation(text=f'Violation Rate: {violation_rate:.2f}% (Expected: 5%)<br>Total Violations: {len(violation_indices)}/{len(returns_eval)}',
                      xref='paper', yref='paper', x=0.02, y=0.98,
                      showarrow=False, bgcolor='rgba(255,255,255,0.8)', bordercolor='black', borderwidth=1)
    
    fig.update_layout(
        title=f'{model} - VaR Backtest Pipeline - {dataset}',
        xaxis_title='Date',
        yaxis_title='Return / VaR',
        hovermode='x unified',
        height=600,
        template='plotly_white'
    )
    fig.show()
    
    # Print summary
    print(f"\n{'='*60}")
    print(f"VaR BACKTEST PIPELINE - {model} on {dataset}")
    print(f"{'='*60}")
    print(f"1. PREDICTED VOL (from model)")
    print(f"   - Mean: {model_preds.mean():.4f}")
    print(f"   - Std:  {model_preds.std():.4f}")
    print(f"\n2. VaR CALCULATION (95% confidence, t-distribution)")
    print(f"   - t_alpha (5% quantile, df={nu}): {t_alpha:.4f}")
    print(f"   - VaR threshold = {t_alpha:.4f} * predicted_vol * sqrt(({nu}-2)/{nu})")
    print(f"   - Mean VaR threshold: {var_95_dist.mean():.4f}")
    print(f"\n3. BACKTEST VIOLATIONS")
    print(f"   - Actual violations: {len(violation_indices)} / {len(returns_eval)}")
    print(f"   - Violation rate: {violation_rate:.2f}%")
    print(f"   - Expected rate: 5%")
    print(f"   - Status: {'✓ PASS' if 4 < violation_rate < 6 else '✗ FAIL'}")
    print(f"{'='*60}\n")

model_opts = sorted(preds_df['Model'].unique())
dataset_opts = sorted(preds_df['Dataset'].unique())
interact(plot_var_backtest, model=Dropdown(options=model_opts, value=model_opts[0]), dataset=Dropdown(options=dataset_opts, value=dataset_opts[0]))

interactive(children=(Dropdown(description='model', options=('GARCH', 'GJR-GARCH', 'LSTM', 'LSTMGARCH', 'Trans…

<function __main__.plot_var_backtest(model, dataset)>